## Initialization

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal
from random import sample

import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve
from data_processing.types import NasaGenerationSettings

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
# def moving_average(arr, n=5):
#     ret = np.cumsum(arr, dtype=float)
#     ret[n:] = ret[n:] - ret[:-n]
#     mov_avg = ret[n-1:] / n
#     prefix = np.empty((n-1,))
#     prefix[:] = np.nan
#     return np.concatenate((prefix, mov_avg))


# def moving_average_centered(arr, n=5):
#     if n % 2 != 1:
#         raise ValueError("Centered moving average needs odd window size")
#     prefix_count = (n-1)//2
#     ret = np.nancumsum(arr, dtype=float)
#     ret[n:] = ret[n:] - ret[:-n]
#     mov_avg = ret[n-1:] / n
#     prefix = np.empty((prefix_count,))
#     suffix = np.empty((prefix_count,))
#     prefix[:] = np.nan
#     suffix[:] = np.nan
#     return np.concatenate((prefix, mov_avg, suffix))

## Experiment ID Input

In [ ]:
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

In [ ]:
# experiment_ids = input_experiment_ids()

In [ ]:
# # more here?
# experiment_neutron_data: ExperimentNeutronData = {
#     exp_id: {}
#     for exp_id in experiment_ids
# }

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )

# is_new_calibration = calib_input.lower() == "y"
# calibrated_energy_column: EnergyColumn = (
#     DetectorDataframeColumn.RECALIBRATED_ENERGY
#     if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
# )
# calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
# strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
# factory_fn = make_strategy_factory_fn(
#     strategy_factory, "nasa", False, settings)
# experiment_neutron_data = make_strategy_for_experiments(
#     experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

## Pulse Selection

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    
    gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    neutrons_only = psd_report.query(n_class_col_name).copy()
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
def correct_raw_signals(
    raw_signals_df: pd.DataFrame,
    baseline_idx_range: int = 30,
    max_adc: int = 16367,
    baseline_offset: float = 0.10
) -> pd.DataFrame:
    # offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()
    # baselines = signals_np[:, :baseline_idx_range].mean(axis=1).reshape(-1, 1)
    # signals_np = -signals_np + baselines + offset
    signals_np = -signals_np + max_adc
    corrected_signals = pd.DataFrame(
        signals_np,
        index=raw_signals_df.index,
        columns=raw_signals_df.columns
    )
    return corrected_signals

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    signals_df = exp_data["signals_df"]
    
    neutron_signals = signals_df.loc[neutrons_only.index].astype("int32")
    gamma_signals = signals_df.loc[gamma_only.index].astype("int32")

    # n_signals_np = neutron_signals.to_numpy()
    # n_baselines = n_signals_np.max(axis=1).reshape(-1, 1)
    # n_signals_np = -n_signals_np + n_baselines
    # print(n_signals_np.max())
    # neutron_signals = pd.DataFrame(n_signals_np, index=neutron_signals.index, columns=neutron_signals.columns)
    neutron_signals = correct_raw_signals(neutron_signals)

    # g_signals_np = gamma_signals.to_numpy()
    # g_baselines = g_signals_np.max(axis=1).reshape(-1, 1)
    # g_signals_np = -g_signals_np + g_baselines
    # print(g_signals_np.max())
    # gamma_signals = pd.DataFrame(g_signals_np, index=gamma_signals.index, columns=gamma_signals.columns)
    gamma_signals = correct_raw_signals(gamma_signals)

    exp_data["neutron_signals"] = neutron_signals
    exp_data["gamma_signals"] = gamma_signals

In [ ]:
min_height = 13000
max_height = 14500

for exp_id, exp_data in experiment_neutron_data.items():
    neutron_signals = exp_data["neutron_signals"]
    gamma_signals = exp_data["gamma_signals"]

    selected_neutron = None
    selected_gamma = None

    bad_neutrons = [64413, 67314, 125305, 127752]
    bad_gamma = []
    
    for neutron_id, neutron_signal in neutron_signals.iterrows():
        # get clean neutron pulse (no secondary peak)
        if selected_neutron is not None:
            break
        if neutron_id in bad_neutrons:
            continue
        n_height = neutron_signal.max()
        if n_height < min_height or n_height > max_height:
            continue
        # peaks, peak_data = find_peaks(neutron_signal, height=200, prominence=50)
        # filtered_peaks = [peak for peak in peaks if abs(peak - 50) > 15]
        # if len(filtered_peaks) == 0:
        else:
            # get matching clean gamma pulse
            for gamma_id, gamma_signal in gamma_signals.iterrows():
                if selected_gamma is not None:
                    break
                if gamma_id in bad_gamma:
                    continue
                g_height = gamma_signal.max()
                if abs(g_height - n_height) > (0.01 * n_height):
                    continue
                # peaks, peak_data = find_peaks(gamma_signal, height=200, prominence=50)
                # filtered_peaks = [peak for peak in peaks if abs(peak - 50) > 15]
                # if len(filtered_peaks) > 0:
                #     continue
                print(f"Neutron ID = {neutron_id}, height = {n_height}")
                print(f"Gamma ID = {gamma_id}, height = {g_height}")
                selected_neutron = neutron_id, neutron_signal
                selected_gamma = gamma_id, gamma_signal

    if selected_neutron is None or selected_gamma is None:
        raise Exception("No pulse found")
    exp_data["selected_neutron"] = selected_neutron
    exp_data["selected_gamma"] = selected_gamma

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
def h_arrow(text, x1, x2, y, ax, above=True, scale=20, text_x_offset=0.5, text_y_offset=0.5):
    text_params = {
        "ha": "center",
        "va": "bottom" if above else "top",
        "fontsize": fontsize - 2
    }
    arrow = mpl.patches.FancyArrowPatch(
        (x1, y), (x2, y), arrowstyle="<->", mutation_scale=scale
    )
    ax.add_patch(arrow)
    annotate_y = text_y_offset if above else -text_y_offset
    ax.annotate(
        text, (text_x_offset, annotate_y), xycoords=arrow, **text_params
    )

In [ ]:
def textbox(text, x, y, width, height, facecolor, textcolor, ax, text_x_offset=0.5, text_y_offset=0.5, ha="center"):
    rect_params = {
        "linewidth": 0,
        # "ec": "black",
        "fc": facecolor
    }
    text_params = {
        "ha": ha,
        "va": "center",
        "fontsize": fontsize,
        "color": textcolor
    }
    rect = mpl.patches.Rectangle((x, y), width, height, **rect_params)
    ax.add_patch(rect)
    # poly = mpl.patches.Polygon(
    ax.annotate(
        # (t_2 + t_3) / 2,
        text,
        (text_x_offset, text_y_offset),
        xycoords=rect,
        **text_params
    )

In [ ]:
# 3c CFD trigger (2025-04-24)
trigger_time = t_t = 144
pulse_time = t_omega = 400
max_adc = 16367

for exp_id, exp_data in experiment_neutron_data.items():
    neutron_id, selected_neutron = exp_data["selected_neutron"]
    gamma_id, selected_gamma = exp_data["selected_gamma"]

    neutron_y = selected_neutron.values
    neutron_y = np.pad(neutron_y, 50, mode="edge")
    neutron_x = np.arange(-50, len(neutron_y) - 50) * 2

    fig, ax = plt.subplots(figsize=(14, 12))
    divider = make_axes_locatable(ax)
    axes_height = 0.4
    ax2 = divider.append_axes("bottom", axes_height, pad=0.8, sharex=ax)
    
    ax.plot(neutron_x, neutron_y, label="Neutron", linewidth=3)
    n_boxes = 1
    margin = 0.05
    box_height = (1-(n_boxes-1)*margin)/n_boxes
    textbox("Pre-trigger", 0, 0*(box_height+margin), t_t, box_height, bg_grey, "white", ax2)

    trigger_line_height = 2800
    con = mpl.patches.ConnectionPatch(
        (t_t, 1),
        (t_t, trigger_line_height),
        "data",
        "data",
        axesA=ax2,
        axesB=ax,
        linewidth=2,
        linestyle=":",
        color=bg_red
    )
    ax2.add_artist(con)
    ax.text(t_t+2, trigger_line_height - 275, "CFD Trigger", ha="left", va="center", fontsize=fontsize-2)
    ax.text(pulse_time / 2, max_adc, "Acquisition Window", ha="center", va="bottom", fontsize=fontsize-2)
    box_width_offset = 2
    box_height_offset = 50
    box = mpl.patches.Rectangle(
        (box_width_offset, box_height_offset),
        pulse_time-2*box_width_offset,
        max_adc-2*box_height_offset,
        # (0, 0), pulse_time, max_adc,
        ec="black",
        fc="#f5f5f5",
        # alpha=0.1,
        zorder=0,
        lw=3
    )
    ax.add_patch(box)

    ax.set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.set_xlabel("Time (ns)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax.tick_params(axis="y", labelleft=False)
    ax.set_ylim(0, 17000)
    ax.set_xlim(-100, pulse_time+100)
    # ax.set_xlim(0, t_3)
    # TODO why isn't this removing labels for values outside range?
    ax.xaxis.set_major_formatter(
        lambda x, _: str(int(x)) if x >= 0 and x <= 400 else ""
    )
    ax.yaxis.set_major_formatter(mpl.ticker.NullFormatter())
    for tick in ax.xaxis.get_majorticklabels():
        tick_x, tick_y = tick.get_position()
        if tick_x == 150.0:
            tick.set_ha("left")
    ax.spines["bottom"].set_position("zero")
    ax.spines["left"].set_position("zero")
    # ax.spines["left"].set_linewidth(3)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

    # ax2.xaxis.set_tick_params(
    #     left=False, labelleft=False, bottom=False, labelbottom=False
    # )
    ax2.patch.set_visible(False)
    ax2.axis("off")

In [ ]:
# 3d Charge integration (2025-04-25)
trigger_time = t_t = 144
pulse_time = t_omega = 400
pre_gate = 50
short_gate = 22
gate_width = 250

t_1 = t_t - pre_gate
t_2 = t_1 + short_gate
t_3 = t_1 + gate_width

for exp_id, exp_data in experiment_neutron_data.items():
    neutron_id, selected_neutron = exp_data["selected_neutron"]
    gamma_id, selected_gamma = exp_data["selected_gamma"]

    neutron_y = selected_neutron.values
    gamma_y = selected_gamma.values
    neutron_x = np.arange(0, len(selected_neutron)) * 2
    gamma_x = np.arange(0, len(selected_gamma)) * 2
    # print(len(selected_neutron))

    fig, ax = plt.subplots(figsize=(14, 12))
    divider = make_axes_locatable(ax)
    axes_height = 1.4
    ax2 = divider.append_axes("bottom", axes_height, pad=1.2, sharex=ax)
    
    ax.plot(neutron_x, neutron_y, label="Neutron", linewidth=3)
    # ax.fill_between(
    #     neutron_x, neutron_y,
    #     where=neutron_x < (t_1 - 20), color="lightgrey"
    # )
    # ax.fill_between(
    #     neutron_x, neutron_y,
    #     where=((t_1 <= neutron_x) & (neutron_x < t_3)),
    #     color="#a2c7ff"
    # )
    # ax.fill_between(
    #     neutron_x, neutron_y,
    #     where=((t_2 <= neutron_x) & (neutron_x < t_3)),
    #     color="#8a9fb8"
    # )
    ax.plot(gamma_x, gamma_y, label="Gamma", linewidth=3)
    ax.fill_between(
        neutron_x, neutron_y, gamma_y, where=(100 <= neutron_x), color=bg_blue, interpolate=True
    )

    # h_arrow("Baseline", 0, 75, 500, ax, text_y_offset=0.75)
    # h_arrow("Tail integral", t_2, t_3, 2000, ax, text_y_offset=0.75)
    # h_arrow("Total integral", t_1, t_3, 2750, ax, text_y_offset=0.75)
    n_boxes = 3
    margin = 0.05
    box_height = (1-(n_boxes-1)*margin)/n_boxes
    # box_height = (axes_height-(n_boxes-1)*margin)/n_boxes
    # box_height = 0.2
    print(axes_height)
    print(box_height)
    print(box_height*n_boxes)
    textbox("Short gate (head integral)", t_1, 0, short_gate, box_height, bg_grey, "black", ax2, ha="left", text_x_offset=1.1)
    textbox("Gate (total integral)", t_1, 1*(box_height+margin), gate_width, box_height, bg_grey, "white", ax2)
    textbox("Pre-gate", t_1, 2*(box_height+margin), pre_gate, box_height, bg_grey, "white", ax2)
    # textbox("Pre-trigger", 0, 3*(box_height+margin), t_t, box_height, bg_grey, "white", ax2)
    # textbox("Acquisition window", 0, 4*(box_height+margin), 400, box_height, bg_grey, "white", ax2)

    trigger_line_height = 2800
    con = mpl.patches.ConnectionPatch(
        (t_t, 2*(box_height+margin)),
        (t_t, trigger_line_height),
        "data",
        "data",
        axesA=ax2,
        axesB=ax,
        linewidth=2,
        linestyle=":",
        color=bg_red
    )
    ax2.add_artist(con)
    ax.text(t_t+2, trigger_line_height - 275, "CFD Trigger", ha="left", va="center", fontsize=fontsize-2)

    ax.set_ylim(0, None)
    ax.set_xlim(0, pulse_time)
    
    ax.set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    for tick in ax.xaxis.get_majorticklabels():
        tick_x, tick_y = tick.get_position()
        if tick_x == 150.0:
            tick.set_ha("left")
    # ax.yaxis.set_major_locator(mpl.ticker.LinearLocator())
    ax.yaxis.set_major_formatter(mpl.ticker.NullFormatter())

    sec = ax.secondary_xaxis(location=0)
    sec.set_xticks([t_1, t_2, t_3], labels=["\n$t_1$", "\n$t_2$", "\n$t_3$"])
    sec.tick_params('x', labelsize=fontsize, length=0, pad=10)
    sec.set_xlabel("Time (ns)", fontsize=fontsize)

    # ax2.xaxis.set_tick_params(
    #     left=False, labelleft=False, bottom=False, labelbottom=False
    # )
    ax2.patch.set_visible(False)
    ax2.axis("off")

In [ ]:
trigger_time = t_t = 144
pulse_time = t_omega = 400

cfd_delay = 6  # 6 values = 12 ns
cfd_fraction = 0.5

for exp_id, exp_data in experiment_neutron_data.items():
    neutron_id, selected_neutron = exp_data["selected_neutron"]

    neutron_y = selected_neutron.values
    neutron_x = np.arange(0, len(selected_neutron)) * 2

    first_value = neutron_y[0]
    delay_y = np.empty_like(neutron_y)
    delay_y[:cfd_delay] = first_value
    delay_y[cfd_delay:] = neutron_y[:-cfd_delay]
    # _x = neutron_x.reshape(-1, 1)
    # _y1 = neutron_y.reshape(-1, 1)
    # _y2 = delay_y.reshape(-1, 1)
    # print(np.concat((_x, _y1, _y2), axis=1))
    
    neutron_max = neutron_y.max()
    cfd_max = -neutron_max * cfd_fraction
    invert_y = -neutron_y
    cfd_max_idx = np.argmax(invert_y < cfd_max)
    cfd_indices = np.arange(0, len(neutron_y))
    cfd_mask = cfd_indices > cfd_max_idx
    invert_y[cfd_mask] = 0

    cfd_sum = delay_y + invert_y
    # find argmin
    sum_min_idx = np.argmin(cfd_sum)
    # find argmax
    sum_max_idx = np.argmax(cfd_sum)
    within_idx_range = (cfd_indices >= sum_min_idx) & (cfd_indices <= sum_max_idx)
    # between these indexes, find index where sum is >=0 but last index was <=0
    cross_zero = np.empty_like(cfd_sum)
    cross_zero[1:] = (cfd_sum[:-1] <= 0) & (cfd_sum[1:] >= 0)
    cross_zero[0] = False
    trigger_index = np.argmax(within_idx_range & cross_zero)
    trigger_x = trigger_index * 2

    fig, axs = plt.subplots(
        nrows=3,
        sharex=True,
        figsize=(12, 16)
    )
    fig.subplots_adjust(hspace=0.05)
    ax0, ax1, ax2 = axs

    ax0.plot(neutron_x, neutron_y)
    ax0.plot(neutron_x, delay_y, alpha=0.2, color="red")
    # ax1.plot(neutron_x, neutron_y, alpha=0.2)
    ax1.plot(neutron_x, invert_y)
    ax2.plot(neutron_x, cfd_sum)
    # ax1.set_xlim(90, 120)
    # ax1.set_ylim(10000, None)

    h_arrow("Pre-trigger", 0, t_t, 1500, ax0, text_x_offset=0.4, text_y_offset=1.0)
    h_arrow("Record length", 0, t_omega, 3000, ax0, text_y_offset=1.0)
    h_arrow("CFD delay (12 ns)", 95, 110, 13500, ax0, above=True, scale=10, text_y_offset=1.5)
    ax2.hlines(0, 0, 400, linestyles="dotted")
    ax2.plot(trigger_x, 0, "o", markersize=10)

    ax0.set_ylim(None, 15000)
    # ax1.set_ylim(None, 14000)
    ax1.set_ylim(-15000, None)
    # ax3.set_ylim(-15000, 15000)

    fig.supylabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax2.set_xlabel("Time (ns)", fontsize=fontsize)
    for ax in axs:
        ax.tick_params(labelsize=fontsize)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    pass

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    pass

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    pass

In [ ]:
input("Processing done, hit Enter to finish")
stop()